# 04 - 超参数调优

## 学习目标
- 掌握 GridSearchCV 和 RandomizedSearchCV
- 了解贝叶斯优化

In [ ]:
import numpy as np
import pandas as pd

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import (
    train_test_split, GridSearchCV, RandomizedSearchCV,
    cross_val_score
)
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

## 1. GridSearchCV

In [ ]:
data = fetch_california_housing()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 参数网格
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 20, None],
    'min_samples_split': [2, 5, 10]
}

grid_search = GridSearchCV(
    RandomForestRegressor(random_state=42),
    param_grid,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f'\n最佳参数: {grid_search.best_params_}')
print(f'最佳分数: {-grid_search.best_score_:.4f}')

## 2. RandomizedSearchCV

In [ ]:
from scipy.stats import randint

param_dist = {
    'n_estimators': randint(50, 300),
    'max_depth': [5, 10, 20, None],
    'min_samples_split': randint(2, 20)
}

random_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_dist,
    n_iter=20,
    cv=3,
    scoring='neg_mean_squared_error',
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

print(f'最佳参数: {random_search.best_params_}')
print(f'最佳分数: {-random_search.best_score_:.4f}')

## 3. Pipeline + GridSearch

In [ ]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestRegressor(random_state=42))
])

param_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth': [5, 10]
}

grid = GridSearchCV(pipeline, param_grid, cv=3, n_jobs=-1)
grid.fit(X_train, y_train)

print(f'最佳参数: {grid.best_params_}')
print(f'测试集 R²: {grid.score(X_test, y_test):.4f}')

## 小结
- GridSearch 穷举搜索，适合小空间
- RandomSearch 随机采样，适合大空间
- Pipeline 中用 `step__param` 指定参数